In [2]:
import pandas as pd
import numpy as np
import gc

# --- 1. Carregar os dataframes necessários ---
print("Carregando arquivos...")
try:
    # Arquivo com os pares e a lista de homônimos (do 2.gera_pares.ipynb)
    df_pares = pd.read_pickle("IpumsNORWAY_pares.pkl")

    # Arquivo com as pessoas únicas usadas no experimento (do 2.gera_pares.ipynb)
    df_person_exp = pd.read_csv("IpumsNORWAY_distinct_person.csv")

    # Arquivo com os municípios únicos (do 2.gera_pares.ipynb)
    df_municipio_exp = pd.read_csv("IpumsNORWAY_distinct_municipio.csv")

    # Arquivo com as profissões únicas (do 2.gera_pares.ipynb)
    df_profissao_exp = pd.read_csv("IpumsNORWAY_distinct_profissao.csv")
    
    print("Arquivos carregados com sucesso!")

    # --- 2. Cálculos das Estatísticas ---
    print("Calculando estatísticas...")

    # Qtd pessoas distintas base (Não é possível calcular diretamente sem a base original completa)
    qtd_pessoas_distintas_base = "Não disponível"

    # Qtd pessoas distintas experimento
    qtd_pessoas_distintas_experimento = len(df_person_exp)

    # Quantidade de pares
    quantidade_pares = len(df_pares)

    # Cálculos sobre a lista de homônimos
    # A lista inclui o próprio parente, então subtraímos 1.
    homonimos_count = df_pares['lista_homonimos'].str.len() - 1

    # Media (lista homonimo - 1)
    media_homonimos = homonimos_count.mean()

    # Mediana (lista homonimo - 1)
    mediana_homonimos = homonimos_count.median()

    # Percentil 5 (lista homonimo - 1)
    percentil_5 = homonimos_count.quantile(0.05)

    # Somatório (lista homonimo) - Total de candidatos somando todas as listas
    somatorio_homonimos = df_pares['lista_homonimos'].str.len().sum()

    # Qtd enderecos distintos experimento
    qtd_enderecos_distintos = len(df_municipio_exp)

    # Qtd profissoes distintas experimento
    qtd_profissoes_distintas = len(df_profissao_exp)

    # Desvio Padrão Populacional (lista homonimo - 1)
    desvio_populacional = np.std(homonimos_count)

    # Percentil 90 (lista homonimo - 1)
    percentil_90 = homonimos_count.quantile(0.90)

    # Percentil 10 (lista homonimo - 1)
    percentil_10 = homonimos_count.quantile(0.10)

    # Limpar memória
    del df_pares, df_person_exp, df_municipio_exp, df_profissao_exp, homonimos_count
    gc.collect()

    # --- 3. Apresentação dos Resultados ---

    # Criar um DataFrame para exibir os resultados de forma organizada
    data = {
        'Estatística': [
            'Qtd pessoas distintas base',
            'Qtd pessoas distintas experimento',
            'Quantidade de pares',
            'Media (lista homonimo - 1)',
            'Mediana (lista homonimo - 1)',
            'Percentil 5 (lista homonimo - 1)',
            'Somatorio (lista homonimo)',
            'Qtd enderecos distintos experimento',
            'Qtd profissoes distintas experimento',
            'Desvio populacional (lista homonimo - 1)',
            'Percentil 90 (lista homonimo - 1)',
            'Percentil 10 (lista homonimo - 1)'
        ],
        'Valor': [
            qtd_pessoas_distintas_base,
            f"{qtd_pessoas_distintas_experimento:,}",
            f"{quantidade_pares:,}",
            f"{media_homonimos:.4f}",
            f"{mediana_homonimos:.4f}",
            f"{percentil_5:.4f}",
            f"{somatorio_homonimos:,}",
            f"{qtd_enderecos_distintos:,}",
            f"{qtd_profissoes_distintas:,}",
            f"{desvio_populacional:.4f}",
            f"{percentil_90:.4f}",
            f"{percentil_10:.4f}"
        ]
    }

    df_stats = pd.DataFrame(data)

    # Imprimir o DataFrame formatado para copiar para o Google Sheets
    print("\n--- Copie a tabela abaixo e cole no Google Sheets ---")
    print(df_stats.to_csv(sep='\t', index=False))

except FileNotFoundError as e:
    print(f"Erro: Arquivo não encontrado. Verifique se os notebooks anteriores foram executados.")
    print(f"Detalhe do erro: {e}")

except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

Carregando arquivos...
Arquivos carregados com sucesso!
Calculando estatísticas...

--- Copie a tabela abaixo e cole no Google Sheets ---
Estatística	Valor
Qtd pessoas distintas base	Não disponível
Qtd pessoas distintas experimento	1,395,798
Quantidade de pares	783,882
Media (lista homonimo - 1)	177.7246
Mediana (lista homonimo - 1)	21.0000
Percentil 5 (lista homonimo - 1)	1.0000
Somatorio (lista homonimo)	140,099,006
Qtd enderecos distintos experimento	755
Qtd profissoes distintas experimento	504
Desvio populacional (lista homonimo - 1)	534.0867
Percentil 90 (lista homonimo - 1)	447.0000
Percentil 10 (lista homonimo - 1)	1.0000



In [1]:
import pandas as pd
import numpy as np
import os
import gc

# --- 1. Configuração e Preparação ---
print("Iniciando cálculo de estatísticas dos splits (MODO DE BAIXA MEMÓRIA)...")
splits = range(1, 6)
all_stats = []
master_file = "IpumsNORWAY_pares.pkl"

if not os.path.exists(master_file):
    print(f"\nERRO CRÍTICO: O arquivo '{master_file}' não foi encontrado. Não é possível continuar.")
else:
    # --- 2. Loop para processar cada split ---
    for split in splits:
        print(f"\nProcessando Split {split}...")
        split_path = f"./splits/{split}"
        
        try:
            # Carregar os arquivos do split (estes são menores)
            df_train = pd.read_csv(os.path.join(split_path, "train_pares.csv"))
            df_val = pd.read_csv(os.path.join(split_path, "val_pares.csv"))
            df_test = pd.read_csv(os.path.join(split_path, "test_pares.csv"))

            # Dicionário para armazenar as estatísticas deste split
            stats = {'Split': split}

            # --- Estatísticas de Treino (não precisa do arquivo mestre) ---
            stats['Treino_Qtd data'] = len(df_train)
            stats['Treino_Qtd data pair-wise'] = len(df_train)
            stats['Treino_Qtd data pair-wise label=1'] = (df_train['label'] == 1).sum()
            stats['Treino_Qtd data pair-wise label=0'] = (df_train['label'] == 0).sum()

            # --- Processamento Otimizado para Validação e Teste ---
            # Carrega o arquivo mestre em pedaços para encontrar os dados de Validação
            print(f"  Processando Validação do Split {split}...")
            df_master_chunk = pd.read_pickle(master_file) # Carrega o mestre temporariamente
            df_val_merged = pd.merge(df_val, df_master_chunk[['person_id_chefe', 'person_id_parente', 'lista_homonimos']], left_on=['person_id_chefe', 'person_id_candidato'], right_on=['person_id_chefe', 'person_id_parente'], how='left')
            df_val_merged['len_homonimos'] = df_val_merged['lista_homonimos'].str.len()
            stats['Validacao_Qtd data'] = len(df_val_merged)
            stats['Validacao_Media (lista homonimo)'] = df_val_merged['len_homonimos'].mean()
            stats['Validacao_Desvio (lista homonimo)'] = df_val_merged['len_homonimos'].std()
            
            # Processamento para Teste
            print(f"  Processando Teste do Split {split}...")
            df_test_merged = pd.merge(df_test, df_master_chunk[['person_id_chefe', 'person_id_parente', 'lista_homonimos']], left_on=['person_id_chefe', 'person_id_candidato'], right_on=['person_id_chefe', 'person_id_parente'], how='left')
            df_test_merged['len_homonimos'] = df_test_merged['lista_homonimos'].str.len()
            stats['Teste_Qtd data'] = len(df_test_merged)
            stats['Teste_Media (lista homonimo)'] = df_test_merged['len_homonimos'].mean()
            stats['Teste_Desvio (lista homonimo)'] = df_test_merged['len_homonimos'].std()
            
            del df_master_chunk, df_val_merged, df_test_merged # Libera memória
            gc.collect()

            all_stats.append(stats)
            
            del df_train, df_val, df_test
            gc.collect()

        except FileNotFoundError:
            print(f"  AVISO: Arquivos para o Split {split} não encontrados em '{split_path}'. Pulando este split.")
            continue
        except Exception as e:
            print(f"  ERRO no processamento do Split {split}: {e}")
            continue

    # --- 3. Consolidação e Apresentação dos Resultados ---
    if all_stats:
        df_final = pd.DataFrame(all_stats)
        
        media_row = {col: np.nan for col in df_final.columns}
        media_row['Split'] = 'Media'
        
        for col in df_final.columns:
            if pd.api.types.is_numeric_dtype(df_final[col]):
                media_row[col] = df_final[col].mean()
                
        df_final = pd.concat([df_final, pd.DataFrame([media_row])], ignore_index=True)

        print("\n--- Copie a tabela abaixo e cole no Google Sheets ---")
        print(df_final.to_csv(sep='\t', index=False, float_format='%.2f'))
    else:
        print("\nNenhuma estatística foi calculada.")

Iniciando cálculo de estatísticas dos splits (MODO DE BAIXA MEMÓRIA)...

Processando Split 1...
  Processando Validação do Split 1...
  Processando Teste do Split 1...

Processando Split 2...
  Processando Validação do Split 2...
  Processando Teste do Split 2...

Processando Split 3...
  Processando Validação do Split 3...
  Processando Teste do Split 3...

Processando Split 4...
  Processando Validação do Split 4...
  Processando Teste do Split 4...

Processando Split 5...
  Processando Validação do Split 5...
  Processando Teste do Split 5...

--- Copie a tabela abaixo e cole no Google Sheets ---
Split	Treino_Qtd data	Treino_Qtd data pair-wise	Treino_Qtd data pair-wise label=1	Treino_Qtd data pair-wise label=0	Validacao_Qtd data	Validacao_Media (lista homonimo)	Validacao_Desvio (lista homonimo)	Teste_Qtd data	Teste_Media (lista homonimo)	Teste_Desvio (lista homonimo)
1.00	4758531.00	4758531.00	564395.00	4194136.00	529509.00	177.59	527.16	27943798.00	178.57	538.34
2.00	4759572.00	475

In [6]:
import pandas as pd

try:
    # Carrega o arquivo pkl
    print("Carregando 'IpumsNORWAY_pares.pkl' para inspeção...")
    df_pares_inspecao = pd.read_pickle("IpumsNORWAY_pares.pkl")

    # 1. Mostra a lista de colunas
    print("\n--- COLUNAS DISPONÍVEIS ---")
    print(df_pares_inspecao.columns.tolist())

    # 2. Mostra as 5 primeiras linhas
    print("\n--- 5 PRIMEIRAS LINHAS ---")
    print(df_pares_inspecao.head())

except FileNotFoundError:
    print("Erro: O arquivo 'IpumsNORWAY_pares.pkl' não foi encontrado no diretório.")
except Exception as e:
    print(f"Ocorreu um erro ao ler o arquivo: {e}")


Carregando 'IpumsNORWAY_pares.pkl' para inspeção...

--- COLUNAS DISPONÍVEIS ---
['person_id_chefe', 'person_id_parente', 'RELATED', 'lista_homonimos', 'ID', 'FOLD']

--- 5 PRIMEIRAS LINHAS ---
  person_id_chefe person_id_parente           RELATED  \
0     285464000_1       285464000_5  Biological child   
1      89129000_1        89129000_2            Spouse   
2     284655000_1       284655000_4            Spouse   
3      37881000_1        37881000_2            Spouse   
4      57711000_1        57711000_4  Biological child   

                                     lista_homonimos  ID  FOLD  
0  [129380000_4, 203560000_4, 254099000_1, 260362...   1     1  
1  [3721000_4, 50552000_4, 89129000_2, 118629000_...   2     1  
2  [7701000_2, 249538000_6, 280924000_4, 28092700...   3     1  
3                           [33491000_5, 37881000_2]   4     1  
4              [57711000_4, 99257000_3, 192340000_4]   5     1  


In [4]:
import pandas as pd
from scipy.stats import wilcoxon

# 1. Configurações
splits = [1, 2, 3, 4, 5]
metrics = ["mrr", "recall@1", "recall@5", "recall@10", "recall@15", "recall@20"]
headers = ["MRR", "Recall@1", "Recall@5", "Recall@10", "Recall@15", "Recall@20"]

# Dicionários para guardar os 5 valores (um por split) de cada métrica para cada modelo
xgb_data = {m: [] for m in metrics}
lgbm_data = {m: [] for m in metrics}

# 2. Carregar os dados
print("Carregando resultados e calculando médias por split...")
for split in splits:
    # Ajuste o caminho se a sua pasta output estiver em outro lugar
    df_xgb = pd.read_csv(f"output/{split}/xgb_ranker.csv")
    df_lgbm = pd.read_csv(f"output/{split}/lgbm_ranker.csv")
    
    for m in metrics:
        # Pega a média da métrica naquele split e guarda na lista
        xgb_data[m].append(df_xgb[m].mean())
        lgbm_data[m].append(df_lgbm[m].mean())

# 3. Aplicar o Teste de Wilcoxon e montar as linhas de resultado
w_stats = ["W"]
p_values = ["p_valor"]

for m in metrics:
    # Modelo B (LGBM) vs Modelo A (XGBoost), alternative='greater' (LGBM > XGB)
    stat, p_value = wilcoxon(lgbm_data[m], xgb_data[m], alternative='greater')
    
    # Formatação com 4 casas decimais e trocando ponto por vírgula para o Sheets
    w_stats.append(f"{stat:.4f}".replace(".", ","))
    p_values.append(f"{p_value:.4f}".replace(".", ","))

# 4. Imprimir para o Google Sheets
# Imprime o cabeçalho
print("\t" + "\t".join(headers))
# Imprime o W
print("\t".join(w_stats))
# Imprime o P-Valor
print("\t".join(p_values))

print("\n" + "="*80)
for i, m in enumerate(metrics):
    p = float(p_values[i+1].replace(",", "."))
    if p <= 0.05:
        print(f"{headers[i]}: LGBM foi significativamente melhor! (p={p})")
    else:
        print(f"{headers[i]}: Empate estatístico (sem evidência de que LGBM é melhor). (p={p})")

Carregando resultados e calculando médias por split...
	MRR	Recall@1	Recall@5	Recall@10	Recall@15	Recall@20
W	15,0000	15,0000	15,0000	15,0000	14,0000	15,0000
p_valor	0,0312	0,0312	0,0312	0,0312	0,0625	0,0312

INTERPRETAÇÃO RÁPIDA (Não precisa copiar):
MRR: LGBM foi significativamente melhor! (p=0.0312)
Recall@1: LGBM foi significativamente melhor! (p=0.0312)
Recall@5: LGBM foi significativamente melhor! (p=0.0312)
Recall@10: LGBM foi significativamente melhor! (p=0.0312)
Recall@15: Empate estatístico (sem evidência de que LGBM é melhor). (p=0.0625)
Recall@20: LGBM foi significativamente melhor! (p=0.0312)
